In [ ]:
%pip install -r requirements.txt

Load data

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/Research/PUMA")

#root = Path.cwd()
train_image_dir = root / "Dataset/01_training_dataset_tif_ROIs"
tissue_geojson_dir = root / "Dataset/01_training_dataset_geojson_tissue"
nuclei_geojson_dir = root / "Dataset/01_training_dataset_geojson_nuclei"

print("train_image_dir =", train_image_dir)
print("tissue_geojson_dir =", tissue_geojson_dir)
print("nucleu_geojson_dir =", nuclei_geojson_dir)

Preprocess

In [ ]:
"""
Rare-class-focused preprocessing for PUMA Track 2.

Click-to-run. No argparse.
All paths are relative to root = Path.cwd().

Expected raw folders:
    Dataset/01_training_dataset_tif_ROIs/*.tif
    Dataset/01_training_dataset_geojson_tissue/*_tissue.geojson
    Dataset/01_training_dataset_geojson_nuclei/*_nuclei.geojson

Output folders:
    dataset_processed/images/*.npy
    dataset_processed/tissue_sem/*.npy       PUMA tissue IDs: 0 background, 1..5 tissue
    dataset_processed/nuclei_nc/*.npy        nuclei class IDs: 0..9, 255 non-nucleus
    dataset_processed/nuclei_hv/*.npy        HoVer map [2,H,W]
    dataset_processed/cellpose_flows/*.npy   Cellpose flow [2,H,W]
    dataset_processed/sample_metadata.json   rare-class flags and sampling weights

Important design:
    The model still uses only 5 tissue classes. Background tissue ID 0 is stored here,
    but the dataset converts it to 255 ignore during training.

Rare-class strategy:
    1. Always save the original 1024 sample.
    2. If a raw ROI contains rare tissue/nuclei, create extra rare-centered translated
       crops with suffix __rareXX. This increases the number of useful rare samples.
    3. Also store per-sample rare metadata so train_stage1.py/train_stage2.py can use
       a WeightedRandomSampler.
"""

import json
import math
import os
import random
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import cv2
import numpy as np
import tifffile as tiff
from tqdm import tqdm

# ============================================================
# Click-to-run configuration
# ============================================================

#root = Path.cwd()
raw_dir = root / "Dataset"
out_dir = root / "dataset_processed"

image_dir = raw_dir / "01_training_dataset_tif_ROIs"
tissue_geojson_dir = raw_dir / "01_training_dataset_geojson_tissue"
nuclei_geojson_dir = raw_dir / "01_training_dataset_geojson_nuclei"

image_size = 1024
crop_size = 1024

# Generate real Cellpose flow files during preprocessing.
# Set to False only if you want zero Cellpose flows during training.
generate_cellpose_flows = True
cellpose_model_type = "cyto3"
cellpose_batch_size = 1

# Rare crop generation.
make_rare_centered_crops = True
max_rare_crops_per_image = 3
rare_crop_jitter_px = 96
random_seed = 42

# If True, existing .npy files are kept. Set True for fast resume.
skip_existing = True

# If True, wipe sample_metadata.json and rebuild it from this run.
# The .npy files themselves are not deleted.
rebuild_metadata = True

# PUMA tissue output IDs.
PUMA_TISSUE_NAME_TO_ID = {
    "tissue_stroma": 1,
    "tissue_blood_vessel": 2,
    "tissue_tumor": 3,
    "tissue_epidermis": 4,
    "tissue_necrosis": 5,
}

PUMA_NUCLEI_NAME_TO_ID = {
    "nuclei_tumor": 0,
    "nuclei_lymphocyte": 1,
    "nuclei_plasma_cell": 2,
    "nuclei_histiocyte": 3,
    "nuclei_melanophage": 4,
    "nuclei_neutrophil": 5,
    "nuclei_stroma": 6,
    "nuclei_epithelium": 7,
    "nuclei_endothelium": 8,
    "nuclei_apoptosis": 9,
}

# Rare classes to focus on. These are based on your Stage 1 logs.
RARE_TISSUE_IDS = {
    2: "tissue_blood_vessel",
    4: "tissue_epidermis",
    5: "tissue_necrosis",
}
RARE_NUCLEI_IDS = {
    2: "nuclei_plasma_cell",
    4: "nuclei_melanophage",
    5: "nuclei_neutrophil",
    8: "nuclei_endothelium",
    9: "nuclei_apoptosis",
}

# Higher = stronger oversampling in training.
RARE_TISSUE_SAMPLE_BONUS = {
    2: 3.0,   # blood vessel
    4: 2.0,   # epidermis
    5: 6.0,   # necrosis
}
RARE_NUCLEI_SAMPLE_BONUS = {
    2: 6.0,   # plasma cell
    4: 4.0,   # melanophage
    5: 8.0,   # neutrophil
    8: 5.0,   # endothelium
    9: 8.0,   # apoptosis
}


# ============================================================
# GeoJSON parsing
# ============================================================

def _feature_class_name(feature: dict) -> Optional[str]:
    props = feature.get("properties", {}) or {}
    cls = props.get("classification", {}) or {}
    name = cls.get("name")
    if name is None:
        name = props.get("name") or props.get("class") or props.get("label")
    return name


def _polygon_arrays_from_geometry(geometry: Optional[dict]) -> List[np.ndarray]:
    if not geometry:
        return []
    gtype = geometry.get("type")
    coords = geometry.get("coordinates", [])
    polys: List[np.ndarray] = []

    if gtype == "Polygon":
        if coords:
            arr = np.asarray(coords[0], dtype=np.float32)
            if arr.ndim == 2 and arr.shape[0] >= 3:
                polys.append(arr)
    elif gtype == "MultiPolygon":
        for poly in coords:
            if not poly:
                continue
            arr = np.asarray(poly[0], dtype=np.float32)
            if arr.ndim == 2 and arr.shape[0] >= 3:
                polys.append(arr)
    return polys


def _polygon_arrays_from_multiple_polygons(data: dict) -> Iterable[Tuple[str, np.ndarray]]:
    """Support Grand-Challenge-style multiple-polygon JSON if present."""
    for poly in data.get("polygons", []):
        name = poly.get("name") or poly.get("classification")
        pts = poly.get("path_points") or poly.get("coordinates") or poly.get("points")
        if name is None or pts is None:
            continue
        arr = np.asarray(pts, dtype=np.float32)
        if arr.ndim == 2 and arr.shape[0] >= 3:
            yield name, arr


def parse_geojson_masks(
    geojson_path: Path,
    class_dict: Dict[str, int],
    shape_hw: Tuple[int, int],
    is_instance: bool,
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """Rasterize polygons at the raw image shape."""
    h, w = shape_hw
    background_value = 255 if is_instance else 0
    sem_mask = np.full((h, w), background_value, dtype=np.uint8)
    inst_mask = np.zeros((h, w), dtype=np.int32) if is_instance else None

    if not geojson_path.exists():
        return sem_mask, inst_mask

    with open(geojson_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    inst_id = 1

    if "features" in data:
        for feature in data.get("features", []):
            class_name = _feature_class_name(feature)
            if class_name not in class_dict:
                continue
            class_id = int(class_dict[class_name])
            polygons = _polygon_arrays_from_geometry(feature.get("geometry"))
            for poly in polygons:
                poly_i = np.round(poly).astype(np.int32)
                cv2.fillPoly(sem_mask, [poly_i], color=class_id)
                if is_instance and inst_mask is not None:
                    cv2.fillPoly(inst_mask, [poly_i], color=inst_id)
                    inst_id += 1
    else:
        for class_name, poly in _polygon_arrays_from_multiple_polygons(data):
            if class_name not in class_dict:
                continue
            class_id = int(class_dict[class_name])
            poly_i = np.round(poly).astype(np.int32)
            cv2.fillPoly(sem_mask, [poly_i], color=class_id)
            if is_instance and inst_mask is not None:
                cv2.fillPoly(inst_mask, [poly_i], color=inst_id)
                inst_id += 1

    return sem_mask, inst_mask


# ============================================================
# Mask/flow helpers
# ============================================================

def compute_hv_map(inst_mask: np.ndarray) -> np.ndarray:
    """HoVer-Net style horizontal/vertical maps, shape [2,H,W]."""
    h_map = np.zeros_like(inst_mask, dtype=np.float32)
    v_map = np.zeros_like(inst_mask, dtype=np.float32)

    for inst_id in np.unique(inst_mask):
        if inst_id == 0:
            continue
        ys, xs = np.where(inst_mask == inst_id)
        if len(xs) == 0:
            continue
        x_center = float(xs.mean())
        y_center = float(ys.mean())
        x_radius = max((float(xs.max()) - float(xs.min())) / 2.0, 1.0)
        y_radius = max((float(ys.max()) - float(ys.min())) / 2.0, 1.0)
        h_map[ys, xs] = np.clip((xs - x_center) / (x_radius + 1e-8), -1.0, 1.0)
        v_map[ys, xs] = np.clip((ys - y_center) / (y_radius + 1e-8), -1.0, 1.0)

    return np.stack([h_map, v_map], axis=0).astype(np.float16)


def read_rgb_tif(path: Path) -> np.ndarray:
    image = tiff.imread(str(path))
    if image.ndim == 2:
        image = np.stack([image, image, image], axis=-1)
    if image.ndim == 3 and image.shape[0] in [3, 4] and image.shape[-1] not in [3, 4]:
        image = np.transpose(image, (1, 2, 0))
    if image.shape[-1] == 4:
        image = image[..., :3]
    if image.dtype != np.uint8:
        image = image.astype(np.float32)
        image = image - image.min()
        image = image / (image.max() + 1e-6)
        image = (image * 255.0).clip(0, 255).astype(np.uint8)
    return image


def resize_all(
    image: np.ndarray,
    tissue: np.ndarray,
    nuclei: np.ndarray,
    inst: np.ndarray,
    size: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if image.shape[0] == size and image.shape[1] == size:
        return image, tissue, nuclei, inst
    image_r = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    tissue_r = cv2.resize(tissue, (size, size), interpolation=cv2.INTER_NEAREST)
    nuclei_r = cv2.resize(nuclei, (size, size), interpolation=cv2.INTER_NEAREST)
    inst_r = cv2.resize(inst.astype(np.int32), (size, size), interpolation=cv2.INTER_NEAREST)
    return image_r, tissue_r.astype(np.uint8), nuclei_r.astype(np.uint8), inst_r.astype(np.int32)


def translate_to_center(
    image: np.ndarray,
    tissue: np.ndarray,
    nuclei: np.ndarray,
    inst: np.ndarray,
    center_xy: Tuple[float, float],
    out_size: int,
    jitter_px: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Translate a rare object toward the crop center while keeping 1024x1024 output."""
    h, w = image.shape[:2]
    cx, cy = center_xy
    jx = random.randint(-jitter_px, jitter_px) if jitter_px > 0 else 0
    jy = random.randint(-jitter_px, jitter_px) if jitter_px > 0 else 0
    target_x = out_size / 2.0 + jx
    target_y = out_size / 2.0 + jy
    dx = target_x - cx
    dy = target_y - cy
    matrix = np.array([[1.0, 0.0, dx], [0.0, 1.0, dy]], dtype=np.float32)

    image_t = cv2.warpAffine(
        image,
        matrix,
        (out_size, out_size),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101,
    )
    tissue_t = cv2.warpAffine(
        tissue,
        matrix,
        (out_size, out_size),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )
    nuclei_t = cv2.warpAffine(
        nuclei,
        matrix,
        (out_size, out_size),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,
    )
    # OpenCV warpAffine is not reliable for int32 on all builds.
    # Use float32 with nearest interpolation, then cast back to int32.
    inst_t = cv2.warpAffine(
        inst.astype(np.float32),
        matrix,
        (out_size, out_size),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    return image_t, tissue_t.astype(np.uint8), nuclei_t.astype(np.uint8), np.rint(inst_t).astype(np.int32)


def component_centers(mask: np.ndarray, class_ids: Iterable[int], max_per_class: int = 2) -> List[Tuple[int, Tuple[float, float], int]]:
    """Return class_id, center_xy, area for connected components of selected classes."""
    out: List[Tuple[int, Tuple[float, float], int]] = []
    for cls in class_ids:
        binary = (mask == cls).astype(np.uint8)
        n, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
        comps = []
        for comp_id in range(1, n):
            area = int(stats[comp_id, cv2.CC_STAT_AREA])
            if area <= 0:
                continue
            cx, cy = centroids[comp_id]
            comps.append((area, (float(cx), float(cy))))
        comps.sort(reverse=True, key=lambda x: x[0])
        for area, center in comps[:max_per_class]:
            out.append((int(cls), center, int(area)))
    return out


def sample_weight_from_masks(tissue: np.ndarray, nuclei: np.ndarray, is_rare_augmented: bool) -> Tuple[float, List[int], List[int]]:
    tissue_present = sorted(int(x) for x in np.unique(tissue) if int(x) in RARE_TISSUE_IDS)
    nuclei_present = sorted(int(x) for x in np.unique(nuclei) if int(x) in RARE_NUCLEI_IDS)

    weight = 1.0
    for cls in tissue_present:
        weight += RARE_TISSUE_SAMPLE_BONUS.get(cls, 0.0)
    for cls in nuclei_present:
        weight += RARE_NUCLEI_SAMPLE_BONUS.get(cls, 0.0)
    if is_rare_augmented:
        weight *= 1.5
    return float(weight), tissue_present, nuclei_present


class CellposeFlowGenerator:
    def __init__(self, enabled: bool, model_type: str):
        self.enabled = bool(enabled)
        self.model = None
        if not self.enabled:
            return
        try:
            import torch
            from cellpose import models
            use_gpu = torch.cuda.is_available()
            self.model = models.CellposeModel(gpu=use_gpu, model_type=model_type)
            print(f"[Cellpose] Loaded model_type={model_type} gpu={use_gpu}")
        except Exception as exc:
            print(f"[Cellpose][WARN] Could not load Cellpose. Zero flows will be stored. Error: {exc}")
            self.model = None

    def make_flow(self, image_rgb: np.ndarray) -> np.ndarray:
        h, w = image_rgb.shape[:2]
        if self.model is None:
            return np.zeros((2, h, w), dtype=np.float16)
        try:
            result = self.model.eval(
                image_rgb,
                diameter=None,
                channels=[0, 0],
                flow_threshold=None,
                cellprob_threshold=0.0,
            )
            if len(result) == 4:
                _, flows, _, _ = result
            else:
                _, flows, _ = result
            flow = flows[1] if isinstance(flows, list) and len(flows) > 1 else flows
            flow = np.asarray(flow)
            if flow.ndim == 3 and flow.shape[0] >= 2:
                flow = flow[:2]
            elif flow.ndim == 3 and flow.shape[-1] >= 2:
                flow = flow[..., :2].transpose(2, 0, 1)
            else:
                raise RuntimeError(f"Unexpected Cellpose flow shape: {flow.shape}")
            if flow.shape[1] != h or flow.shape[2] != w:
                flow = np.stack([
                    cv2.resize(flow[0], (w, h), interpolation=cv2.INTER_LINEAR),
                    cv2.resize(flow[1], (w, h), interpolation=cv2.INTER_LINEAR),
                ], axis=0)
            return flow.astype(np.float16)
        except Exception as exc:
            print(f"[Cellpose][WARN] Flow generation failed; storing zero flow. Error: {exc}")
            return np.zeros((2, h, w), dtype=np.float16)


def save_processed_sample(
    base_name: str,
    image: np.ndarray,
    tissue: np.ndarray,
    nuclei: np.ndarray,
    inst: np.ndarray,
    flow_generator: CellposeFlowGenerator,
    metadata: List[dict],
    is_rare_augmented: bool,
    source_name: str,
) -> None:
    paths = {
        "image": out_dir / "images" / f"{base_name}.npy",
        "tissue": out_dir / "tissue_sem" / f"{base_name}.npy",
        "nuclei": out_dir / "nuclei_nc" / f"{base_name}.npy",
        "hv": out_dir / "nuclei_hv" / f"{base_name}.npy",
        "cp": out_dir / "cellpose_flows" / f"{base_name}.npy",
    }

    weight, rare_tissue, rare_nuclei = sample_weight_from_masks(tissue, nuclei, is_rare_augmented)

    if not (skip_existing and all(p.exists() for p in paths.values())):
        hv = compute_hv_map(inst)
        cp_flow = flow_generator.make_flow(image)
        np.save(paths["image"], image.astype(np.uint8))
        np.save(paths["tissue"], tissue.astype(np.uint8))
        np.save(paths["nuclei"], nuclei.astype(np.uint8))
        np.save(paths["hv"], hv)
        np.save(paths["cp"], cp_flow)

    metadata.append({
        "base_name": base_name,
        "source_name": source_name,
        "is_rare_augmented": bool(is_rare_augmented),
        "rare_tissue_ids": rare_tissue,
        "rare_nuclei_ids": rare_nuclei,
        "sample_weight": weight,
    })


def find_annotation_file(folder: Path, base: str, suffix: str) -> Path:
    """Find annotation robustly across common PUMA/QuPath naming variants."""
    candidates = [
        folder / f"{base}_{suffix}.geojson",
        folder / f"{base}.geojson",
        folder / f"{base}-{suffix}.geojson",
        folder / f"{base} {suffix}.geojson",
    ]
    for path in candidates:
        if path.exists():
            return path
    hits = sorted(folder.glob(f"*{base}*{suffix}*.geojson"))
    if hits:
        return hits[0]
    hits = sorted(folder.glob(f"*{base}*.geojson"))
    if hits:
        return hits[0]
    return candidates[0]


def process_one_roi(img_path: Path, flow_generator: CellposeFlowGenerator, metadata: List[dict]) -> None:
    base = img_path.stem
    image = read_rgb_tif(img_path)
    h, w = image.shape[:2]

    tissue_geojson = find_annotation_file(tissue_geojson_dir, base, "tissue")
    nuclei_geojson = find_annotation_file(nuclei_geojson_dir, base, "nuclei")

    tissue, _ = parse_geojson_masks(tissue_geojson, PUMA_TISSUE_NAME_TO_ID, (h, w), is_instance=False)
    nuclei, inst = parse_geojson_masks(nuclei_geojson, PUMA_NUCLEI_NAME_TO_ID, (h, w), is_instance=True)
    assert inst is not None

    image_1024, tissue_1024, nuclei_1024, inst_1024 = resize_all(image, tissue, nuclei, inst, image_size)

    save_processed_sample(
        base_name=base,
        image=image_1024,
        tissue=tissue_1024,
        nuclei=nuclei_1024,
        inst=inst_1024,
        flow_generator=flow_generator,
        metadata=metadata,
        is_rare_augmented=False,
        source_name=base,
    )

    if not make_rare_centered_crops:
        return

    centers: List[Tuple[str, int, Tuple[float, float], int]] = []
    for cls, center, area in component_centers(tissue_1024, RARE_TISSUE_IDS.keys(), max_per_class=2):
        centers.append(("tissue", cls, center, area))
    for cls, center, area in component_centers(nuclei_1024, RARE_NUCLEI_IDS.keys(), max_per_class=3):
        centers.append(("nuclei", cls, center, area))

    if not centers:
        return

    # Prioritize the rarest classes and larger components.
    def priority(item):
        kind, cls, _, area = item
        bonus = RARE_NUCLEI_SAMPLE_BONUS.get(cls, 0.0) if kind == "nuclei" else RARE_TISSUE_SAMPLE_BONUS.get(cls, 0.0)
        return bonus * 100000.0 + area

    centers.sort(key=priority, reverse=True)
    centers = centers[:max_rare_crops_per_image]

    for j, (kind, cls, center, _) in enumerate(centers):
        aug_name = f"{base}__rare{j:02d}_{kind}{cls}"
        im_t, tissue_t, nuclei_t, inst_t = translate_to_center(
            image_1024,
            tissue_1024,
            nuclei_1024,
            inst_1024,
            center_xy=center,
            out_size=image_size,
            jitter_px=rare_crop_jitter_px,
        )
        save_processed_sample(
            base_name=aug_name,
            image=im_t,
            tissue=tissue_t,
            nuclei=nuclei_t,
            inst=inst_t,
            flow_generator=flow_generator,
            metadata=metadata,
            is_rare_augmented=True,
            source_name=base,
        )


def main() -> None:
    random.seed(random_seed)
    np.random.seed(random_seed)

    for subdir in ["images", "tissue_sem", "nuclei_nc", "nuclei_hv", "cellpose_flows"]:
        (out_dir / subdir).mkdir(parents=True, exist_ok=True)

    img_files = sorted(image_dir.glob("*.tif")) + sorted(image_dir.glob("*.tiff"))
    if not img_files:
        raise FileNotFoundError(f"No TIFF files found in {image_dir}")

    print(f"[Root] {root}")
    print(f"[Raw] {raw_dir}")
    print(f"[Output] {out_dir}")
    print(f"[Images] {len(img_files)}")
    print(f"[Rare crops] enabled={make_rare_centered_crops}, max_per_image={max_rare_crops_per_image}")
    print(f"[Cellpose] generate_cellpose_flows={generate_cellpose_flows}")

    flow_generator = CellposeFlowGenerator(generate_cellpose_flows, cellpose_model_type)
    metadata: List[dict] = []

    for img_path in tqdm(img_files, desc="Preprocess rare-focused"):
        process_one_roi(img_path, flow_generator, metadata)

    metadata_path = out_dir / "sample_metadata.json"
    if rebuild_metadata or not metadata_path.exists():
        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2)

    n_rare_aug = sum(1 for m in metadata if m["is_rare_augmented"])
    print("\n[Done]")
    print(f"Processed samples in metadata: {len(metadata)}")
    print(f"Rare augmented samples: {n_rare_aug}")
    print(f"Metadata: {metadata_path}")
    print("Stored PUMA tissue IDs in tissue_sem; dataset converts background 0 to ignore 255.")


if __name__ == "__main__":
    main()


Train Stage 1

In [ ]:
# Clean up VRAM
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
"""
Rare-focused Stage 1 training.

Click-to-run. No argparse.
All paths are relative to root = Path.cwd().

Main rare-class improvements:
    - Rare-centered crops from preprocess.py are used automatically.
    - WeightedRandomSampler oversamples rare-class samples.
    - FN-focused Focal Tversky loss is blended in smoothly after warmup.
    - Class weights strongly emphasize rare tissue/nuclei classes.
    - Checkpoint selection is rare-focused through utils/metrics.py.
    - Best epoch is printed at the end.
    - SC-DFA and spatial prior are also ramped smoothly to avoid training shocks.
    - Checkpoints are safely saved via local /content then copied to Drive.
"""

import shutil
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
try:
    import bitsandbytes as bnb
except Exception:
    bnb = None

from dataloaders import (
    INTERNAL_TISSUE_ID_TO_NAME,
    PUMA_NUCLEI_ID_TO_NAME,
    PUMADataset,
    get_train_transforms,
    get_val_transforms,
)
from models import UnifiedPanopticNet, get_cnn_spatial_prior
from train import train_one_epoch, validate
from utils import MultiTaskUncertaintyLoss, PUMAMetrics
from utils.split_utils import make_or_load_group_split


# ============================================================
# Click-to-run config
# ============================================================

#root = Path.cwd()

data_dir = root / "dataset_processed"
uni_weight_dir = root
checkpoint_dir = root / "checkpoints"
split_file = checkpoint_dir / "split_seed42.npz"

image_size = 1024
stride = 768
batch_size = 16
epochs = 50
num_workers = 2
seed = 42
val_ratio = 0.2
force_new_split = False
val_original_only = True

lr = 1e-4
weight_decay = 1e-4

# Smooth schedule. Do not abruptly switch difficult losses/modules on.
focal_start_epoch = 10
focal_full_epoch = 16
focal_max_weight = 0.5

sc_dfa_start_epoch = 15
sc_dfa_full_epoch = 22
sc_dfa_max_weight = 0.3

prior_start_epoch = 20
prior_full_epoch = 28
prior_max_weight = 0.2

# Used only as fallback. PUMADataset reads site type from file name.
default_site_type = "metastatic"

# You said you have Cellpose and will use it. Keep this at 0.0 so training always
# uses the stored Cellpose flow files generated by preprocess.py.
zero_cellpose_prob = 0.0

# Oversample rare images more than once per epoch.
samples_per_epoch_multiplier = 1.0

multi_gpu = False
use_fp16 = True

# ---------------------------------------------------------
# CHANGED: Point resume to the last saved checkpoint
# ---------------------------------------------------------
resume = checkpoint_dir / "puma_epoch_last_s1.pth"

# Class weights are repeated in checkpoint metadata for transparency.
tissue_class_weights = [1.0, 4.0, 0.8, 3.0, 7.0]
nuclei_class_weights = [0.8, 1.0, 7.0, 2.5, 4.5, 8.0, 2.0, 2.5, 5.5, 8.0]


def linear_ramp(epoch, start, end, max_value):
    """Return a smooth linear warmup value for epoch-level schedules."""
    if epoch < start:
        return 0.0
    if epoch >= end:
        return float(max_value)
    progress = (epoch - start + 1) / max(end - start + 1, 1)
    return float(max_value) * float(progress)


def apply_smooth_schedule(model, criterion, epoch):
    """Apply smooth Stage 1 schedule for rare semantic loss, SC-DFA, and prior."""
    core = model.module if hasattr(model, "module") else model

    focal_weight = linear_ramp(epoch, focal_start_epoch, focal_full_epoch, focal_max_weight)
    sc_dfa_weight = linear_ramp(epoch, sc_dfa_start_epoch, sc_dfa_full_epoch, sc_dfa_max_weight)
    prior_weight = linear_ramp(epoch, prior_start_epoch, prior_full_epoch, prior_max_weight)

    if hasattr(criterion, "set_focal_tversky_weight"):
        criterion.set_focal_tversky_weight(focal_weight)
    else:
        criterion.focal_tversky_weight = focal_weight

    if hasattr(core, "set_sc_dfa_lambda"):
        core.set_sc_dfa_lambda(sc_dfa_weight)
    elif hasattr(core, "enable_sc_dfa"):
        core.enable_sc_dfa(sc_dfa_weight > 0.0)

    if hasattr(core, "set_spatial_prior_lambda"):
        core.set_spatial_prior_lambda(prior_weight)

    print(
        f"[SmoothSchedule] epoch={epoch:03d} "
        f"focal={focal_weight:.3f} "
        f"sc_dfa={sc_dfa_weight:.3f} "
        f"prior={prior_weight:.3f}"
    )
    return focal_weight, sc_dfa_weight, prior_weight


def as_config_namespace():
    return SimpleNamespace(
        image_size=image_size,
        stride=stride,
        default_site_type=default_site_type,
    )


def safe_torch_save(obj, path):
    """Avoid Google Drive partial-write corruption for large checkpoints."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if Path("/content").exists():
        local_tmp = Path("/content") / (path.name + ".tmp")
        local_final = Path("/content") / path.name
    else:
        local_tmp = path.with_suffix(path.suffix + ".tmp")
        local_final = path

    torch.save(obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)

    if local_final != local_tmp:
        local_tmp.replace(local_final)

    if local_final != path:
        shutil.copy2(local_final, path)
        _ = torch.load(path, map_location="cpu", weights_only=False)

    print(f"[Checkpoint] Saved and verified: {path}")


def make_inference_config(model):
    core = model.module if hasattr(model, "module") else model
    return {
        "architecture": "merged_v22_architecture_v4_labels_no_tissue_background_rare_focused",
        "image_size": image_size,
        "tile_size": image_size,
        "stride": stride,
        "num_tissue_classes": 5,
        "num_nuclei_classes": 10,
        "use_sc_dfa": bool(core.use_sc_dfa),
        "lambda_sc_dfa": float(getattr(core, "lambda_sc_dfa", 0.0)),
        "lambda_prior": float(core.lambda_prior),
        "default_site_type": default_site_type,
        "internal_tissue_id_to_name": INTERNAL_TISSUE_ID_TO_NAME,
        "puma_nuclei_id_to_name": PUMA_NUCLEI_ID_TO_NAME,
        "tissue_internal_to_puma_rule": "puma_id = internal_id + 1; no model background channel",
        "normalization_mean": [0.485, 0.456, 0.406],
        "normalization_std": [0.229, 0.224, 0.225],
        "cellpose_mode_at_training": "real_cellpose_flows_from_dataset_processed_cellpose_flows",
        "zero_cellpose_prob": zero_cellpose_prob,
        "rare_focused_training": True,
        "batch_size": batch_size,
        "epochs": epochs,
        "tissue_class_weights": tissue_class_weights,
        "nuclei_class_weights": nuclei_class_weights,
        "samples_per_epoch_multiplier": samples_per_epoch_multiplier,
        "max_sample_weight": 15.0,
        "smooth_stage1_schedule": {
            "focal_start_epoch": focal_start_epoch,
            "focal_full_epoch": focal_full_epoch,
            "focal_max_weight": focal_max_weight,
            "sc_dfa_start_epoch": sc_dfa_start_epoch,
            "sc_dfa_full_epoch": sc_dfa_full_epoch,
            "sc_dfa_max_weight": sc_dfa_max_weight,
            "prior_start_epoch": prior_start_epoch,
            "prior_full_epoch": prior_full_epoch,
            "prior_max_weight": prior_max_weight,
        },
        "stage1_checkpoint_name": "puma_epoch_best_s1.pth",
        "stage2_checkpoint_name": "nuclei_refiner_residual_best.pth",
        "split_is_group_based": True,
        "validation_original_only": val_original_only,
    }


def save_checkpoint(path, model, criterion, optimizer, scheduler, scaler, epoch, best_score, val_report):
    core = model.module if hasattr(model, "module") else model
    payload = {
        "epoch": int(epoch),
        "model_state": core.state_dict(),
        "criterion_state": criterion.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "best_score": float(best_score),
        "best_val_report": val_report,
        "inference_config": make_inference_config(core),
    }
    safe_torch_save(payload, path)


def make_rare_weighted_sampler(dataset, indices):
    weights = np.asarray(dataset.compute_sample_weights(indices), dtype=np.float64)
    #weights = np.clip(weights, 1.0, 40.0)
    weights = np.clip(weights, 1.0, 15.0)
    num_samples = int(round(len(indices) * samples_per_epoch_multiplier))
    num_samples = max(num_samples, len(indices))
    print(
        f"[Sampler] rare weighted sampler: n_indices={len(indices)} "
        f"num_samples={num_samples} min_w={weights.min():.2f} "
        f"mean_w={weights.mean():.2f} max_w={weights.max():.2f}"
    )
    return WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=num_samples,
        replacement=True,
    )


def print_report(epoch, train_loss, val):
    tissue_names = [INTERNAL_TISSUE_ID_TO_NAME[i] for i in range(5)]
    nuclei_names = [PUMA_NUCLEI_ID_TO_NAME[i] for i in range(10)]
    print("\n" + "=" * 88)
    print(
        f"Epoch {epoch:03d} | train_loss={train_loss:.4f} | "
        f"val_loss={val.get('val_loss', 0):.4f} | selection={val.get('selection_score', 0):.4f} | "
        f"rare={val.get('rare_macro_dice', 0):.4f}"
    )
    print("-" * 88)
    print(f"Tissue loss={val.get('loss_tissue', 0):.4f}")
    for i, name in enumerate(tissue_names):
        print(f"  {i}: {name:<22} dice={val.get(f'tissue_dice_{i}', 0):.4f} iou={val.get(f'tissue_iou_{i}', 0):.4f}")
    print(f"Nuclei loss={val.get('loss_nc', 0):.4f}")
    for i, name in enumerate(nuclei_names):
        print(f"  {i}: {name:<22} dice={val.get(f'nuclei_dice_{i}', 0):.4f} iou={val.get(f'nuclei_iou_{i}', 0):.4f}")
    print(f"NP loss={val.get('loss_np', 0):.4f} | HV loss={val.get('loss_hv', 0):.4f}")
    print(
        f"avg_tissue={val.get('avg_tissue_dice', 0):.4f} | "
        f"avg_nuclei={val.get('avg_nuclei_dice', 0):.4f} | "
        f"rare_tissue={val.get('rare_tissue_macro_dice', 0):.4f} | "
        f"rare_nuclei={val.get('rare_nuclei_macro_dice', 0):.4f}"
    )
    print("=" * 88 + "\n")


def main():
    torch.manual_seed(seed)
    np.random.seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    print(f"[Root] {root}")
    print(f"[Data] {data_dir}")
    print(f"[Checkpoints] {checkpoint_dir}")
    print(f"[Config] batch_size={batch_size} epochs={epochs} zero_cellpose_prob={zero_cellpose_prob}")

    train_ds = PUMADataset(
        data_dir,
        transforms=get_train_transforms(image_size),
        zero_cellpose_prob=zero_cellpose_prob,
    )
    val_ds = PUMADataset(
        data_dir,
        transforms=get_val_transforms(image_size),
        zero_cellpose_prob=0.0,
    )

    split_meta = train_ds.get_split_metadata()
    train_idx, val_idx = make_or_load_group_split(
        source_names=split_meta["source_names"],
        is_original=split_meta["is_original"],
        split_path=split_file,
        seed=seed,
        train_fraction=1.0 - val_ratio,
        force_new=force_new_split,
        val_original_only=val_original_only,
    )
    print(f"[Split] train={len(train_idx)} val={len(val_idx)} file={split_file}")
    print("[Split] Leakage-safe: all rare crops stay with their source image; validation uses originals only.")

    train_loader = DataLoader(
        Subset(train_ds, train_idx),
        batch_size=batch_size,
        sampler=make_rare_weighted_sampler(train_ds, train_idx),
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        Subset(val_ds, val_idx),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
    )

    cnn = get_cnn_spatial_prior(pretrained=True)
    model = UnifiedPanopticNet(
        vit_model=uni_weight_dir,
        cnn_model=cnn,
        num_tissue=5,
        num_nuclei=10,
        load_uni_weights=True,
    ).to(device)

    if multi_gpu and torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)

    criterion = MultiTaskUncertaintyLoss(
        tissue_weights=torch.tensor(tissue_class_weights, dtype=torch.float32),
        nuclei_weights=torch.tensor(nuclei_class_weights, dtype=torch.float32),
    ).to(device)
    params = list(model.parameters()) + list(criterion.parameters())

    if bnb is not None and device.type == "cuda":
        try:
            optimizer = bnb.optim.AdamW8bit(params, lr=lr, weight_decay=weight_decay)
        except Exception:
            optimizer = optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.AdamW(params, lr=lr, weight_decay=weight_decay)

    total_steps = epochs * max(len(train_loader), 1)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and use_fp16)
    metrics = PUMAMetrics()

    # ---------------------------------------------------------
    # CHANGED: Full state-resume logic applied here
    # ---------------------------------------------------------
    best_score = -1.0
    best_epoch = 0
    best_val_report = None
    start_epoch = 1

    if resume is not None:
        resume_path = Path(resume)
        if resume_path.is_file():
            print(f"[INFO] Resuming from checkpoint: {resume_path}")
            checkpoint = torch.load(resume_path, map_location=device, weights_only=False)

            # Load model state
            core = model.module if hasattr(model, "module") else model
            core.load_state_dict(checkpoint["model_state"])

            # Load criterion state (loss function variables)
            if "criterion_state" in checkpoint and checkpoint["criterion_state"] is not None:
                criterion.load_state_dict(checkpoint["criterion_state"], strict=False)

            # Load optimizer state (momentum, gradients context)
            if "optimizer_state" in checkpoint and checkpoint["optimizer_state"] is not None:
                optimizer.load_state_dict(checkpoint["optimizer_state"])

            # Load scheduler state (current LR step)
            if "scheduler_state" in checkpoint and checkpoint["scheduler_state"] is not None:
                scheduler.load_state_dict(checkpoint["scheduler_state"])

            # Load scaler state (for mixed precision stability)
            if "scaler_state" in checkpoint and checkpoint["scaler_state"] is not None:
                scaler.load_state_dict(checkpoint["scaler_state"])

            # Load epoch tracking
            start_epoch = checkpoint["epoch"] + 1
            best_score = checkpoint.get("best_score", -1.0)
            best_val_report = checkpoint.get("best_val_report", None)

            print(f"[INFO] Resumed successfully. Starting from epoch {start_epoch}, current best score: {best_score:.4f}")
        else:
            print(f"[WARN] Resume path {resume_path} does not exist. Starting from scratch.")

    # ---------------------------------------------------------
    # CHANGED: Range now starts precisely at 'start_epoch'
    # ---------------------------------------------------------
    for epoch in range(start_epoch, epochs + 1):
        core = model.module if hasattr(model, "module") else model

        apply_smooth_schedule(model, criterion, epoch)

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scheduler,
            device,
            scaler,
            epoch,
        )
        val = validate(model, val_loader, criterion, metrics, device, epoch)
        print_report(epoch, train_loss, val)

        score = float(val.get("selection_score", -val.get("val_loss", 1e9)))
        if score > best_score:
            best_score = score
            best_epoch = epoch
            best_val_report = dict(val)
            save_path = checkpoint_dir / "puma_epoch_best_s1.pth"
            save_checkpoint(save_path, model, criterion, optimizer, scheduler, scaler, epoch, best_score, best_val_report)
            print(f"Saved best full checkpoint: {save_path} | epoch={best_epoch} score={best_score:.4f}")

        # ---------------------------------------------------------
        # CHANGED: Save verified last checkpoint EVERY epoch so
        # you won't lose progression. The global best_score/report
        # is also passed so it survives the resume process correctly.
        # ---------------------------------------------------------
        last_path = checkpoint_dir / "puma_epoch_last_s1.pth"
        save_checkpoint(last_path, model, criterion, optimizer, scheduler, scaler, epoch, best_score, best_val_report)

    print("\n" + "=" * 88)
    print(f"Stage 1 complete. Best epoch used as checkpoint: {best_epoch}")
    print(f"Best selection score: {best_score:.4f}")
    if best_val_report is not None:
        print(f"Best rare macro dice: {best_val_report.get('rare_macro_dice', 0):.4f}")
        print(f"Best rare tissue dice: {best_val_report.get('rare_tissue_macro_dice', 0):.4f}")
        print(f"Best rare nuclei dice: {best_val_report.get('rare_nuclei_macro_dice', 0):.4f}")
    print(f"Best checkpoint: {checkpoint_dir / 'puma_epoch_best_s1.pth'}")
    print("=" * 88)


if __name__ == "__main__":
    main()

Train Stage 2

In [ ]:
# Clean up VRAM
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [ ]:
"""
Rare-focused Stage 2 residual nuclei refiner.

Click-to-run. No argparse.
All paths are relative to root = Path.cwd().

Leakage control:
    Uses the exact same group-based split file as Stage 1.
    Train includes rare-centered crops from train source images only.
    Validation uses original validation images only.
"""

import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from tqdm import tqdm

try:
    import bitsandbytes as bnb
except Exception:
    bnb = None

from dataloaders import PUMA_NUCLEI_ID_TO_NAME, PUMADataset, get_train_transforms, get_val_transforms
from models import ResidualNucleiRefinerUNet, UnifiedPanopticNet, build_stage2_input, get_cnn_spatial_prior
from utils import PUMAMetrics
from utils.losses import FocalTverskyLoss, SafeCrossEntropyLoss
from utils.split_utils import make_or_load_group_split


# ============================================================
# Click-to-run config
# ============================================================

#root = Path.cwd()

data_dir = root / "dataset_processed"
uni_weight_dir = root
checkpoint_dir = root / "checkpoints"
split_file = checkpoint_dir / "split_seed42.npz"
stage1_ckpt = checkpoint_dir / "puma_epoch_best_s1.pth"

image_size = 1024
batch_size = 16
epochs = 30
num_workers = 2
seed = 42
train_fraction = 0.8
force_new_split = False
val_original_only = True

lr = 1e-4
weight_decay = 1e-4

default_site_type = "metastatic"
use_fp16 = True
resume = None

num_nuclei_classes = 10
stage2_in_channels = 21
ignore_index = 255

rare_nuclei_ids = [2, 4, 5, 8, 9]
nuclei_weights = [0.6, 0.9, 9.0, 2.5, 5.0, 10.0, 2.0, 2.5, 6.0, 10.0]

samples_per_epoch_multiplier = 2.5
kd_temperature = 2.0
keep_lambda_start = 0.80
keep_lambda_end = 0.15
keep_lambda_decay_epochs = 30
alpha_start = 0.05
alpha_end = 0.45
alpha_warmup_epochs = 30


def safe_torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if Path("/content").exists():
        local_tmp = Path("/content") / (path.name + ".tmp")
        local_final = Path("/content") / path.name
    else:
        local_tmp = path.with_suffix(path.suffix + ".tmp")
        local_final = path

    torch.save(obj, local_tmp)
    _ = torch.load(local_tmp, map_location="cpu", weights_only=False)

    if local_final != local_tmp:
        local_tmp.replace(local_final)

    if local_final != path:
        shutil.copy2(local_final, path)
        _ = torch.load(path, map_location="cpu", weights_only=False)

    print(f"[Checkpoint] Saved and verified: {path}")


def load_large_checkpoint(path, device):
    """Copy Drive checkpoint to local /content before loading, avoiding Drive FUSE seek issues."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    if str(path).startswith("/content/drive") and Path("/content").exists():
        local_dir = Path("/content/checkpoints")
        local_dir.mkdir(parents=True, exist_ok=True)
        local_path = local_dir / path.name
        if (not local_path.exists()) or local_path.stat().st_size != path.stat().st_size:
            print(f"[Checkpoint] Copying Stage 1 checkpoint to local runtime: {local_path}")
            shutil.copy2(path, local_path)
        path = local_path

    return torch.load(path, map_location=device, weights_only=False)


def extract_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ["model_state", "model_state_dict", "state_dict"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                checkpoint = checkpoint[key]
                break
    if not isinstance(checkpoint, dict):
        raise ValueError("Unsupported checkpoint format")
    return {k.replace("module.", "", 1): v for k, v in checkpoint.items()}


def alpha_schedule(epoch):
    ratio = min(max(epoch / float(max(alpha_warmup_epochs, 1)), 0.0), 1.0)
    return alpha_start + ratio * (alpha_end - alpha_start)


def keep_lambda_schedule(epoch):
    ratio = min(max(epoch / float(max(keep_lambda_decay_epochs, 1)), 0.0), 1.0)
    return keep_lambda_start + ratio * (keep_lambda_end - keep_lambda_start)


def masked_kl_preservation_loss(refined_logits, s1_logits, targets, temperature=2.0, ignore_index=255):
    valid = targets != ignore_index
    if not torch.any(valid):
        return refined_logits.sum() * 0.0
    log_p_refined = F.log_softmax(refined_logits / temperature, dim=1)
    p_s1 = F.softmax(s1_logits.detach() / temperature, dim=1)
    kl = F.kl_div(log_p_refined, p_s1, reduction="none").sum(dim=1)
    return kl[valid].mean() * (temperature ** 2)


def make_rare_weighted_sampler(dataset, indices):
    weights = np.asarray(dataset.compute_sample_weights(indices), dtype=np.float64)
    weights = np.clip(weights, 1.0, 50.0)
    num_samples = int(round(len(indices) * samples_per_epoch_multiplier))
    num_samples = max(num_samples, len(indices))
    print(
        f"[Sampler] Stage 2 rare weighted sampler: n_indices={len(indices)} "
        f"num_samples={num_samples} min_w={weights.min():.2f} "
        f"mean_w={weights.mean():.2f} max_w={weights.max():.2f}"
    )
    return WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=num_samples,
        replacement=True,
    )


def compute_stage2_scores(metrics_calc, s1_metrics, s2_metrics):
    out = {}
    s1_dice = []
    s2_dice = []
    s1_rare = []
    s2_rare = []

    for k in range(num_nuclei_classes):
        s1_val = s1_metrics.get(f"s1_nuclei_dice_{k}", math.nan)
        s2_val = s2_metrics.get(f"s2_nuclei_dice_{k}", math.nan)
        out[f"s1_nuclei_dice_{k}"] = s1_val
        out[f"s1_nuclei_iou_{k}"] = s1_metrics.get(f"s1_nuclei_iou_{k}", math.nan)
        out[f"s2_nuclei_dice_{k}"] = s2_val
        out[f"s2_nuclei_iou_{k}"] = s2_metrics.get(f"s2_nuclei_iou_{k}", math.nan)
        s1_dice.append(s1_val)
        s2_dice.append(s2_val)

    for k in rare_nuclei_ids:
        s1_rare.append(s1_metrics.get(f"s1_nuclei_dice_{k}", math.nan))
        s2_rare.append(s2_metrics.get(f"s2_nuclei_dice_{k}", math.nan))

    s1_macro = metrics_calc._nanmean(s1_dice)
    s2_macro = metrics_calc._nanmean(s2_dice)
    s1_rare_macro = metrics_calc._nanmean(s1_rare)
    s2_rare_macro = metrics_calc._nanmean(s2_rare)

    out["s1_macro_dice"] = s1_macro
    out["s2_macro_dice"] = s2_macro
    out["s1_rare_macro_dice"] = s1_rare_macro
    out["s2_rare_macro_dice"] = s2_rare_macro
    out["selection_score"] = 0.25 * metrics_calc._nan_to_zero(s2_macro) + 0.75 * metrics_calc._nan_to_zero(s2_rare_macro)
    out["improvement_score"] = 0.25 * (metrics_calc._nan_to_zero(s2_macro) - metrics_calc._nan_to_zero(s1_macro)) + 0.75 * (metrics_calc._nan_to_zero(s2_rare_macro) - metrics_calc._nan_to_zero(s1_rare_macro))
    out["beats_stage1"] = out["improvement_score"] > 0.0
    return out


def fmt(v):
    try:
        v = float(v)
    except Exception:
        return "N/A"
    return "N/A" if math.isnan(v) else f"{v:.4f}"


def print_report(epoch, train_loss, val_loss, results):
    print("\n" + "=" * 92)
    print(f"Stage 2 epoch {epoch:03d} | train={train_loss:.4f} val={val_loss:.4f} alpha={results['alpha']:.3f} keep={results['keep_lambda']:.3f}")
    print("-" * 92)
    for k in range(num_nuclei_classes):
        name = PUMA_NUCLEI_ID_TO_NAME[k]
        s1 = results.get(f"s1_nuclei_dice_{k}")
        s2 = results.get(f"s2_nuclei_dice_{k}")
        delta = math.nan if s1 is None or s2 is None or math.isnan(float(s1)) or math.isnan(float(s2)) else float(s2) - float(s1)
        print(f"{k:02d} {name:<22} S1={fmt(s1):<8} S2={fmt(s2):<8} Δ={fmt(delta):<8}")
    print(f"S1 macro={fmt(results.get('s1_macro_dice'))} | S2 macro={fmt(results.get('s2_macro_dice'))}")
    print(f"S1 rare ={fmt(results.get('s1_rare_macro_dice'))} | S2 rare ={fmt(results.get('s2_rare_macro_dice'))}")
    print(f"selection={fmt(results.get('selection_score'))} improvement={fmt(results.get('improvement_score'))} beats_stage1={results.get('beats_stage1')}")
    print("=" * 92 + "\n")


def main():
    torch.manual_seed(seed)
    np.random.seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.benchmark = True
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    print(f"[Root] {root}")
    print(f"[Data] {data_dir}")
    print(f"[Stage 1 checkpoint] {stage1_ckpt}")
    print(f"[Checkpoints] {checkpoint_dir}")

    train_ds = PUMADataset(data_dir, transforms=get_train_transforms(image_size), zero_cellpose_prob=0.0)
    val_ds = PUMADataset(data_dir, transforms=get_val_transforms(image_size), zero_cellpose_prob=0.0)

    split_meta = train_ds.get_split_metadata()
    train_idx, val_idx = make_or_load_group_split(
        source_names=split_meta["source_names"],
        is_original=split_meta["is_original"],
        split_path=split_file,
        seed=seed,
        train_fraction=train_fraction,
        force_new=force_new_split,
        val_original_only=val_original_only,
    )
    print(f"[Split] train={len(train_idx)} val={len(val_idx)} file={split_file}")
    print("[Split] Leakage-safe: all rare crops stay with their source image; validation uses originals only.")

    train_loader = DataLoader(
        Subset(train_ds, train_idx),
        batch_size=batch_size,
        sampler=make_rare_weighted_sampler(train_ds, train_idx),
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        Subset(val_ds, val_idx),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
    )

    model_s1 = UnifiedPanopticNet(
        vit_model=uni_weight_dir,
        cnn_model=get_cnn_spatial_prior(pretrained=False),
        num_tissue=5,
        num_nuclei=10,
        load_uni_weights=False,
    ).to(device)

    ckpt_s1 = load_large_checkpoint(stage1_ckpt, device)
    model_s1.load_state_dict(extract_state_dict(ckpt_s1), strict=True)

    cfg_s1 = ckpt_s1.get("inference_config", {}) if isinstance(ckpt_s1, dict) else {}
    model_s1.enable_sc_dfa(bool(cfg_s1.get("use_sc_dfa", True)))
    model_s1.set_spatial_prior_lambda(float(cfg_s1.get("lambda_prior", 1.0)))
    model_s1.eval()
    for p in model_s1.parameters():
        p.requires_grad = False

    model_s2 = ResidualNucleiRefinerUNet(
        in_channels=stage2_in_channels,
        out_classes=num_nuclei_classes,
    ).to(device)

    class_weights = torch.tensor(nuclei_weights, dtype=torch.float32, device=device)
    ce_loss = SafeCrossEntropyLoss(weight=class_weights, ignore_index=ignore_index)
    ft_loss = FocalTverskyLoss(
        alpha=0.20,
        beta=0.80,
        gamma=1.60,
        class_weights=class_weights,
        ignore_index=ignore_index,
    ).to(device)
    ce_loss = ce_loss.to(device)

    if bnb is not None and device.type == "cuda":
        try:
            optimizer = bnb.optim.AdamW8bit(model_s2.parameters(), lr=lr, weight_decay=weight_decay)
        except Exception:
            optimizer = optim.AdamW(model_s2.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.AdamW(model_s2.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and use_fp16)
    metrics_calc = PUMAMetrics()

    best_score = -1.0
    best_epoch = 0
    best_improvement = -999.0

    if resume is not None:
        print(f"[WARN] resume path is set to {resume}, but exact resume loading is not implemented in this click-to-run file.")

    for epoch in range(1, epochs + 1):
        alpha = alpha_schedule(epoch)
        keep_lambda = keep_lambda_schedule(epoch)
        model_s2.train()
        train_loss_sum = 0.0

        for batch in tqdm(train_loader, desc=f"Train Stage2 {epoch:03d}", leave=False):
            images = batch["image"].to(device, non_blocking=True)
            targets_nc = batch["nuclei_nc"].to(device, non_blocking=True)
            cellpose_flows = batch["cellpose_flow"].to(device, non_blocking=True)
            site_types = batch.get("site_type") or [default_site_type] * images.shape[0]

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=device.type == "cuda" and use_fp16):
                with torch.no_grad():
                    preds_s1 = model_s1(images, cellpose_flows, site_types)
                    s1_nc_logits = preds_s1["nc"].detach()
                    s2_input = build_stage2_input(images, preds_s1)
                delta_nc = model_s2(s2_input)
                refined_nc = s1_nc_logits + alpha * delta_nc
                loss_ce = ce_loss(refined_nc, targets_nc)
                loss_ft = ft_loss(refined_nc, targets_nc)
                loss_keep = masked_kl_preservation_loss(
                    refined_nc,
                    s1_nc_logits,
                    targets_nc,
                    temperature=kd_temperature,
                    ignore_index=ignore_index,
                )
                loss = loss_ce + loss_ft + keep_lambda * loss_keep

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_s2.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss_sum += float(loss.detach().item())

        scheduler.step()
        avg_train_loss = train_loss_sum / max(len(train_loader), 1)

        model_s2.eval()
        val_loss_sum = 0.0
        s1_acc = metrics_calc.new_semantic_accumulator(num_nuclei_classes, "s1_nuclei", ignore_index=ignore_index, device=device)
        s2_acc = metrics_calc.new_semantic_accumulator(num_nuclei_classes, "s2_nuclei", ignore_index=ignore_index, device=device)

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Valid Stage2 {epoch:03d}", leave=False):
                images = batch["image"].to(device, non_blocking=True)
                targets_nc = batch["nuclei_nc"].to(device, non_blocking=True)
                cellpose_flows = batch["cellpose_flow"].to(device, non_blocking=True)
                site_types = batch.get("site_type") or [default_site_type] * images.shape[0]

                with torch.amp.autocast("cuda", enabled=device.type == "cuda" and use_fp16):
                    preds_s1 = model_s1(images, cellpose_flows, site_types)
                    s1_nc_logits = preds_s1["nc"]
                    s2_input = build_stage2_input(images, preds_s1)
                    delta_nc = model_s2(s2_input)
                    refined_nc = s1_nc_logits + alpha * delta_nc
                    loss_ce = ce_loss(refined_nc, targets_nc)
                    loss_ft = ft_loss(refined_nc, targets_nc)
                    loss_keep = masked_kl_preservation_loss(
                        refined_nc,
                        s1_nc_logits,
                        targets_nc,
                        temperature=kd_temperature,
                        ignore_index=ignore_index,
                    )
                    val_loss = loss_ce + loss_ft + keep_lambda * loss_keep

                val_loss_sum += float(val_loss.detach().item())
                s1_acc.update(s1_nc_logits, targets_nc)
                s2_acc.update(refined_nc, targets_nc)

        avg_val_loss = val_loss_sum / max(len(val_loader), 1)
        results = compute_stage2_scores(metrics_calc, s1_acc.compute(), s2_acc.compute())
        results["alpha"] = alpha
        results["keep_lambda"] = keep_lambda
        print_report(epoch, avg_train_loss, avg_val_loss, results)

        ckpt_payload = {
            "model_state": model_s2.state_dict(),
            "epoch": epoch,
            "alpha": alpha,
            "keep_lambda": keep_lambda,
            "selection_score": results["selection_score"],
            "improvement_score": results["improvement_score"],
            "beats_stage1": results["beats_stage1"],
            "config": {
                "in_channels": stage2_in_channels,
                "out_classes": num_nuclei_classes,
                "residual": True,
                "alpha_start": alpha_start,
                "alpha_end": alpha_end,
                "alpha_warmup_epochs": alpha_warmup_epochs,
                "kd_temperature": kd_temperature,
                "keep_lambda_start": keep_lambda_start,
                "keep_lambda_end": keep_lambda_end,
                "nuclei_weights": nuclei_weights,
                "rare_nuclei_ids": rare_nuclei_ids,
                "uses_5_tissue_probs_no_background": True,
                "stage2_input_channels": stage2_in_channels,
                "split_is_group_based": True,
                "validation_original_only": val_original_only,
            },
        }

        if epoch % 5 == 0 or epoch == epochs:
            safe_torch_save(ckpt_payload, checkpoint_dir / "nuclei_refiner_residual_last.pth")

        score = float(results["selection_score"])
        if score > best_score:
            best_score = score
            best_epoch = epoch
            best_improvement = float(results["improvement_score"])
            safe_torch_save(ckpt_payload, checkpoint_dir / "nuclei_refiner_residual_best.pth")
            print(f"Saved Stage 2 best: epoch={best_epoch} score={best_score:.4f} improvement={best_improvement:+.4f}")

        if not results["beats_stage1"]:
            print("[WARN] Stage 2 has not beaten Stage 1 yet. For Docker inference, prefer Stage 1-only or validate hybrid before enabling Stage 2.\n")

    print("\n" + "=" * 92)
    print(f"Stage 2 complete. Best epoch used as checkpoint: {best_epoch}")
    print(f"Best score: {best_score:.4f}")
    print(f"Best improvement over Stage 1: {best_improvement:+.4f}")
    print(f"Best checkpoint: {checkpoint_dir / 'nuclei_refiner_residual_best.pth'}")
    print("=" * 92)


if __name__ == "__main__":
    main()
